<img src="https://viagemeturismo.abril.com.br/wp-content/uploads/2022/01/GettyImages-478641555-e1642187749880.jpg?quality=90&strip=info&w=1024" alt="Alternative text" width="1000" height="200"/>

Este estudo aprofundado, utilizando o poder da linguagem Python e bibliotecas como Pandas, Matplotlib, NumPy, Geopandas e Folium, vasculha os dados de mortes violentas no Estado da Bahia, buscando compreender a dinâmica desse triste fenômeno e apresentar insights valiosos para subsidiar ações de combate e prevenção.

###### Fontes:
* [Portal Dados Abertos Bahia](https://dados.ba.gov.br/dataset/morte_violenta_estado/resource/6637f86b-9d3e-425f-b92a-afd30a8ba797)
* [IBGE](https://www.ibge.gov.br/cidades-e-estados/ba.html)

 

###### Bibliotecas utilizadas:

* pandas 🐼
* matplotlib 📊
* numpy 🔢
* Geopandas 🗺
* Folium 🌍

<a target="_blank" href="https://colab.research.google.com/github/https://colab.research.google.com/github/erivelton-jr/analise-mortes-violentas-bahia/blob/main/CrimeData_Analysis_Complete.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

### 1. Importando Bibliotecas e Carregando Dados:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv('mortes_violentas_estado.csv')
df

### 2. Explorando os Dados

In [ ]:
df.nunique()

In [ ]:
df.REGIAO.unique()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

##### Com essas informação sabemos que:
* Temos dados de mais de 80% dos municipios da Bahia
* Temos 3 regiões: Interior, Salvador, RMS(Região Metropolitana de Salvador).
* Não temos valores nulos. =D

In [ ]:
#renomeando colunas para melhorar a visualização dos dados
meses = {
    1: 'Janeiro',
    2: 'Fevereiro',
    3: 'Março',
    4: 'Abril',
    5: 'Maio',
    6: 'Junho',
    7: 'Julho',
    8: 'Agosto',
    9: 'Setembro',
    10: 'Outubro',
    11: 'Novembro',
    12: 'Dezembro'
}

df['MES_NOME'] = df['MES'].map(meses)


In [ ]:
#ordenando os meses

ordem_meses = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho', 
               'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']

df['MES_NOME'] = pd.Categorical(df['MES_NOME'], categories=ordem_meses, ordered=True)

In [ ]:
#renomeando a sigla 'RMS' (Região Metropolitana de Salvador) para 'Metropole' para facilitar leitura.
df['REGIAO'].replace(['RMS'], ['Metropole'], inplace=True)

In [ ]:
df.isnull().sum()

### 3. Visualização Temporal

 

In [ ]:
mortes_por_mes = df.groupby('MES_NOME')['QT_VITIMAS'].sum().reset_index()
mortes_por_mes

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(mortes_por_mes['MES_NOME'], mortes_por_mes['QT_VITIMAS'], marker='o', linestyle='-', color='r')

# Adicione rótulos e título
plt.xlabel('Mês')
plt.ylabel('Quantidade de Vítimas')
plt.title('Quantidade de Vítimas por Mês')

for i, (x, y) in enumerate(zip(mortes_por_mes['MES_NOME'], mortes_por_mes['QT_VITIMAS'])):
    plt.annotate(y, (x, y), textcoords="offset points", xytext=(0,5), ha='center')

plt.axhline(
    mortes_por_mes['QT_VITIMAS'].mean(),
    color='b',
    linestyle='--',
    linewidth=2,
    label='Média de vitimas por mês')

plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

A partir desses dados entendemos que durante maior parte do primeiro semestre o Estado da Bahia teve um alto indice de mortes violentas, superando até mesmo a média anual.

### 4. Análise por Região

In [ ]:
mortes_por_regiao = df.groupby(['REGIAO'])['QT_VITIMAS'].sum().reset_index()

In [ ]:
plt.figure(figsize=(6, 3))
plt.bar(mortes_por_regiao['REGIAO'], mortes_por_regiao['QT_VITIMAS'])

plt.show()

### 5. Correlações e Análises Estatísticas:

Agora que sabemos o numero de vitimas de cada região, iremos comparar os dados com a densidade populacional de cada região.
Para isso vamos extrair esses dados do site do [IBGE(Instituto Brasileiro de Geografia e Estatísticas)](https://www.ibge.gov.br/cidades-e-estados/ba.html).

In [ ]:
#extraindo a população de cada municipio

ibge = pd.read_excel('dados_ibge.xlsx')
ibge

In [ ]:
ibge.info()

Como visto acima, os dados do IBGE possui algumas linhas que não precisaremos utilizar. Então vamos verificar quantas linhas são para que possamos removê-las.

#### Tratando os dados do IBGE

In [ ]:
ibge.tail(14)

In [ ]:
#removendo linhas que não iremos utilizar

ibge.drop(index=[x for x in range(419,433)], inplace=True) 
ibge.drop(index=[0, 1], inplace=True)

In [ ]:
ibge.columns

In [ ]:
#removendo colunas que não iremos utilizar

ibge.drop(columns=['Unnamed: 2', 'Unnamed: 3','Unnamed: 4',
                   'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
                   'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12'], inplace=True)

In [ ]:
ibge.columns

In [ ]:
#renomeando colunas
ibge.columns = ['MUNICIPIO','ID_MUNICIPIO', 'POPULACAO']

In [ ]:
ibge.head()

In [ ]:
ibge.shape

In [ ]:
ibge.isnull().sum()

  Agora que fizemos o tratamento dos dados, sabemos que os DataFrame possui:
* Dados de todos os 417 municipios da Bahia.
* Não possui dados nulos.

Também podemos perceber que o `ID_MUNICIPIO` dos dados do IBGE, é identico ao `ID_MUNICIPIO` dos dados de mortes violentas, porém o o `ID_MUNICIPIO` do `ibge` tem um numero a mais. Então iremos corrigir isso.

In [ ]:
ibge['ID_MUNICIPIO'] = ibge['ID_MUNICIPIO'].apply(lambda x: (str(x)[:-1]))

#convertendo tipo dos dados
ibge['ID_MUNICIPIO'] = ibge['ID_MUNICIPIO'].astype('int64')

In [ ]:
ibge.head()

In [ ]:
#unindo dados

municipio = df.groupby('ID_MUNICIPIO')[['MUNICIPIO', 'REGIAO']].first()
populacao = pd.merge(municipio, ibge, on='ID_MUNICIPIO', suffixes=('_remove', ''))

populacao.drop('MUNICIPIO_remove', axis=1, inplace=True)

In [ ]:
populacao.head()

In [ ]:

# Calculando a quantidade total de vítimas por região
vitimas_por_regiao = df.groupby('REGIAO')['QT_VITIMAS'].sum()

# Calculando a população total por região
populacao_por_regiao = populacao.groupby('REGIAO')['POPULACAO'].sum()

# Criando uma figura
plt1 = plt.bar(populacao_por_regiao.index, populacao_por_regiao.values, 
               color='blue', label='População')
plt2  =plt.bar(vitimas_por_regiao.index, vitimas_por_regiao.values, 
               color='red', label='Vitimas', edgecolor='w')

plt.bar_label(plt1, fmt='%i', label_type='center', color='w')
plt.bar_label(plt2,label_type='center', color='w')
plt.yscale('log')

plt.show()

In [ ]:
populacao_por_regiao

Podemos perceber que no interior há uma população maior e com isso também veio a maior taxa de mortes violentas também.

#### Distribuição geográfica de mortes violentas

In [ ]:
!pip install geopandas
import geopandas as gpd

In [ ]:
# criando mapa da bahia

bahia = gpd.read_file('mapa/BA_Municipios_2022.shp')
bahia.head()

In [ ]:
#tratando os dados
bahia['CD_MUN'] = bahia['CD_MUN'].apply(lambda x: (str(x)[:-1]))
bahia['CD_MUN'] = bahia['CD_MUN'].astype('int64')

In [ ]:
#unindo dados

mortes_por_municipio = df.groupby(['ID_MUNICIPIO','MUNICIPIO'])['QT_VITIMAS'].sum().reset_index()

mortes_por_municipio = mortes_por_municipio.merge(bahia, how='inner', left_on='ID_MUNICIPIO', right_on='CD_MUN', sort=True)

#transformando dados em GeoDataFrame

mortes_por_municipio = gpd.GeoDataFrame(mortes_por_municipio)

In [ ]:
#instalando biblioteca folium
!pip install folium

In [ ]:
#importando bibliotecas para plottar o mapa
from branca.colormap import linear
import folium 

In [ ]:
#criando cores

colormap = linear.YlOrRd_07.scale(mortes_por_municipio['QT_VITIMAS'].min(),
                               mortes_por_municipio['QT_VITIMAS'].max())

#Criando um dicionário para mapear o nome do município para a quantidade de vítimas
color_dict = mortes_por_municipio.set_index('MUNICIPIO')['QT_VITIMAS']

In [ ]:
#Criando mapa
mapa = folium.Map([-12.52, -41.69], zoom_start=6)

folium.GeoJson(
    mortes_por_municipio,
    name="Quantidade de Vítimas",
    style_function=lambda feature: {
        "fillColor": colormap(color_dict[feature["properties"]["MUNICIPIO"]]),
        "color": "black",
        "weight": 1,
        "dashArray": "5, 5",
        "fillOpacity": 0.9,
    },
    tooltip=folium.features.GeoJsonTooltip(fields=['MUNICIPIO', 'QT_VITIMAS'],
                                            aliases=['Município', 'Quantidade de Vítimas'],
                                            localize=True)
).add_to(mapa)

# Adicionando legenda
colormap.caption = 'Quantidade de Vítimas'
colormap.add_to(mapa)

# Exibindo o mapa
mapa

Podemos visualizar de forma mapeada onde mais se concentrou as mortes violentas no ano de 2023. Agora vamos aproveitar e comparar a população dos municipios com o maior indice de mortes violentas.

In [ ]:
#10 municipios com o maior numero de mortes violentas

mortes_por_municipio.sort_values(by='QT_VITIMAS', ascending=False).head(10)

In [ ]:
#unindo os dados
morte_populacao = mortes_por_municipio.merge(ibge, how='inner', left_on='ID_MUNICIPIO', 
                                             right_on='ID_MUNICIPIO', sort=True)

#separando colunas que iremos utilizar
morte_populacao = morte_populacao[['ID_MUNICIPIO', 'NM_MUN', 'POPULACAO', 'QT_VITIMAS']].sort_values(by='QT_VITIMAS', ascending=False).head(10)

In [ ]:
morte_populacao

In [ ]:
#criando gráficos para comparar população x mortes
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(morte_populacao.NM_MUN))
width= 0.35

bar1 = ax.bar(x - width / 2, morte_populacao['QT_VITIMAS'], width=width, label='Quantidade de Vítimas')
bar2 = ax.bar(x + width / 2, morte_populacao['POPULACAO'], width=width, label='População')


ax.set_ylabel('População/QT. Vitimas')
ax.set_title('Quantidade de Vítimas x População por Município')
ax.set_xticks([i + width / 2 for i in x])
ax.set_xticklabels(morte_populacao['NM_MUN'], rotation=45, ha='right')
ax.legend()
ax.bar_label(bar1)
ax.bar_label(bar2, fmt='%i')


plt.yscale('log')
plt.tight_layout()
plt.show()

Podemos vê que as cidades com maior indice de mortes violentas também possuia o maior numero de habitantes.

### 6. Análise de Causas

In [ ]:
gr_natureza = df.groupby('GR_NATUREZA')['QT_VITIMAS'].sum().reset_index()
df.loc[df['GR_NATUREZA'] == "HOMICÍDIO DOLOSO COM INDÍCIO DE EXCLUDENTE DE ILICITUDE",
'GR_NATUREZA'] = "HOMICÍDIO DOLOSO (LEGITIMA DEFESA)"

In [ ]:
#criando grafico interativo com plotly
import plotly.express as px

fig = px.pie(df, values='QT_VITIMAS', names='GR_NATUREZA', 
             title='Proporção de Vítimas por Tipo de Crime', 
             color_discrete_sequence=px.colors.qualitative.Set3)

# Mostrar o gráfico
fig.show()

### Conclusão

##### _"A Bahia teve queda de 4,1% no número de mortes violentas em 2023, mas segue como estado com maior registro de mortes violentas no Brasil pelo 5º ano seguido."_ -  [G1](https://g1.globo.com/ba/bahia/noticia/2024/03/12/monitor-da-violencia-2023-bahia.ghtml)

----
#### Análise Temporal

> **Primeiro semestre**: Aumento preocupante das mortes, exigindo medidas imediatas de segurança pública e ações sociais.

> **Segundo semestre**: Redução gradual, mas ainda em patamares inaceitáveis. É fundamental manter o foco no combate à violência, sem se acomodar com a queda.

> **Tendências por Mês**: Auxiliam na alocação estratégica de recursos de segurança e ações preventivas ao longo do ano.

#### Análise por Região:

> **Interior**: Maior número de mortes, exigindo atenção redobrada das autoridades. Investimento em policiamento ostensivo e ações sociais direcionadas são essenciais.

> **Região Metropolitana de Salvador**: Segundo maior índice de mortes, reforçando a necessidade de políticas públicas específicas para a região.

> **Salvador**: Apesar de apresentar o menor número de mortes em relação à população, ainda registra valores alarmantes. Campanhas de conscientização e medidas de segurança específicas para a capital são cruciais.

#### Considerações Adicionais:

>**Fatores socioeconômicos**: Analisá-los em conjunto com os dados de violência é crucial para identificar as raízes do problema e propor soluções eficazes.

>**Mapas de Calor**: Utilizar o Geopandas e o Folium para criar mapas de calor, visualizando a distribuição espacial das mortes e direcionando ações de forma mais precisa.

>**Trabalho em Conjunto**: Governos, entidades da sociedade civil e a comunidade em geral devem unir forças para combater a violência de forma eficaz e duradoura.

A luta contra a violência na Bahia é um desafio árduo, mas não impossível. Através da análise de dados, do investimento em políticas públicas abrangentes e da união de todos os setores da sociedade, podemos construir um futuro mais seguro e pacífico para todos.

*Os dados apresentados neste estudo servem como base para análises mais aprofundadas e a tomada de decisões estratégicas. O combate à violência exige um esforço contínuo e multifacetado, com o monitoramento constante dos indicadores e a adaptação das ações às novas realidades.*

Juntos, podemos construir uma Bahia livre da violência 🔵🔴⚪️ 